# Data Cleaning for E-commerce Cohort Analysis and Time Series Forecasting

## Overview

This notebook performs comprehensive data cleaning on the online retail dataset to prepare it for:
- Cohort analysis to calculate Customer Lifetime Value (LTV)
- Time series forecasting for sales prediction
- ROMI (Return on Marketing Investment) optimization

## Data Cleaning Steps

1. **Data Loading**: Load data from Excel file
2. **Initial Exploration**: Check data structure and basic statistics
3. **Missing Values Handling**: Identify and handle missing data
4. **Data Type Corrections**: Ensure proper data types
5. **Duplicate Removal**: Remove duplicate transactions
6. **Outlier Detection and Treatment**: Handle anomalous values
7. **Data Validation**: Business rule checks
8. **Feature Engineering**: Create derived features for analysis
9. **Final Dataset Preparation**: Save cleaned data for modeling

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Data Loading
print("Loading data from Excel file...")

# Load data from Excel file
excel_file = 'data/online_retail_II.xlsx'

try:
    # Read both sheets
    df_2009_2010 = pd.read_excel(excel_file, sheet_name='Year 2009-2010', engine='openpyxl')
    df_2010_2011 = pd.read_excel(excel_file, sheet_name='Year 2010-2011', engine='openpyxl')

    print(f"✓ Successfully loaded data:")
    print(f"  - 2009-2010: {df_2009_2010.shape[0]:,} rows, {df_2009_2010.shape[1]} columns")
    print(f"  - 2010-2011: {df_2010_2011.shape[0]:,} rows, {df_2010_2011.shape[1]} columns")

    # Combine datasets
    df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
    print(f"  - Combined: {df.shape[0]:,} rows, {df.shape[1]} columns")

except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

Loading data from Excel file...
✓ Successfully loaded data:
  - 2009-2010: 525,461 rows, 8 columns
  - 2010-2011: 541,910 rows, 8 columns
  - Combined: 1,067,371 rows, 8 columns


In [4]:
# Initial Data Exploration
print("=== INITIAL DATA EXPLORATION ===")

# Basic information
print(f"\nDataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Column names: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")

# Check for missing values
print(f"\nMissing values check:")
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    for col, count in missing_counts[missing_counts > 0].items():
        print(f"  {col}: {count:,} missing ({count/len(df)*100:.1f}%)")
else:
    print("  No missing values found")

print("\n✓ Initial exploration completed")

=== INITIAL DATA EXPLORATION ===

Dataset shape: 1,067,371 rows × 8 columns
Column names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Missing values check:
  Description: 4,382 missing (0.4%)
  Customer ID: 243,007 missing (22.8%)

✓ Initial exploration completed


In [5]:
# Data Cleaning: Missing Values Handling
print("=== MISSING VALUES HANDLING ===")

# Check missing values again
print("Missing values before cleaning:")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

# Handle missing Customer ID - these are crucial for cohort analysis
print(f"\nRecords with missing Customer ID: {df['Customer ID'].isnull().sum()}")
print(".1f")

# For cohort analysis, we need Customer ID, so we'll drop records without it
df_clean = df.dropna(subset=['Customer ID']).copy()
print(f"After dropping missing Customer ID: {df_clean.shape[0]:,} rows")

# Handle missing Description - fill with 'Unknown' or infer from StockCode
missing_desc = df_clean['Description'].isnull().sum()
if missing_desc > 0:
    print(f"\nMissing descriptions: {missing_desc}")
    df_clean['Description'] = df_clean['Description'].fillna('Unknown Product')
    print("Filled missing descriptions with 'Unknown Product'")

# Check for any other missing values
remaining_missing = df_clean.isnull().sum().sum()
print(f"\nTotal remaining missing values: {remaining_missing}")

print(f"\nFinal shape after missing value handling: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

=== MISSING VALUES HANDLING ===
Missing values before cleaning:
Description      4382
Customer ID    243007
dtype: int64

Records with missing Customer ID: 243007
.1f
After dropping missing Customer ID: 824,364 rows

Total remaining missing values: 0

Final shape after missing value handling: 824,364 rows × 8 columns


In [6]:
# Data Type Corrections
print("=== DATA TYPE CORRECTIONS ===")

print("Data types before correction:")
print(df_clean.dtypes)

# Convert Customer ID to integer (it's currently float due to NaN handling)
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

# Convert InvoiceDate to datetime if not already
if not pd.api.types.is_datetime64_any_dtype(df_clean['InvoiceDate']):
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
    print("✓ Converted InvoiceDate to datetime")

# Convert Invoice to string (invoice numbers)
df_clean['Invoice'] = df_clean['Invoice'].astype(str)

# Convert StockCode to string
df_clean['StockCode'] = df_clean['StockCode'].astype(str)

# Ensure Quantity is integer
df_clean['Quantity'] = df_clean['Quantity'].astype(int)

# Ensure Price is float
df_clean['Price'] = df_clean['Price'].astype(float)

print("\nData types after correction:")
print(df_clean.dtypes)

# Validate date range
print(f"\nDate range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
print(f"Total days: {(df_clean['InvoiceDate'].max() - df_clean['InvoiceDate'].min()).days} days")

=== DATA TYPE CORRECTIONS ===
Data types before correction:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Data types after correction:
Invoice                   str
StockCode                 str
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Total days: 738 days


In [7]:
# Duplicate Removal
print("=== DUPLICATE REMOVAL ===")

# Check for duplicates
duplicate_count = df_clean.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    duplicate_percentage = (duplicate_count / len(df_clean)) * 100
    print(".3f")

    # Show sample duplicates
    print("\nSample duplicate rows:")
    display(df_clean[df_clean.duplicated()].head())

    # Remove duplicates
    df_clean = df_clean.drop_duplicates()
    print(f"\nAfter removing duplicates: {df_clean.shape[0]:,} rows")
else:
    print("✓ No duplicate rows found")

# Check for duplicate invoices with different details (potential data quality issues)
# This might indicate data entry errors
invoice_duplicates = df_clean.groupby('Invoice').size()
invoices_with_multiple_entries = invoice_duplicates[invoice_duplicates > 1]
print(f"\nInvoices with multiple entries: {len(invoices_with_multiple_entries)}")

if len(invoices_with_multiple_entries) > 0:
    print("Sample invoices with multiple entries:")
    display(invoices_with_multiple_entries.head())

=== DUPLICATE REMOVAL ===
Duplicate rows found: 26479
.3f

Sample duplicate rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom



After removing duplicates: 797,885 rows

Invoices with multiple entries: 37421
Sample invoices with multiple entries:


Invoice
489434     8
489435     4
489436    19
489437    23
489438    17
dtype: int64

In [9]:
# Outlier Detection and Treatment
print("=== OUTLIER DETECTION AND TREATMENT ===")

# Check for negative quantities (returns/cancellations)
negative_qty = df_clean[df_clean['Quantity'] < 0]
print(f"Records with negative quantity: {len(negative_qty)}")
print(".1f")

if len(negative_qty) > 0:
    print("\nSample negative quantity records:")
    print(negative_qty[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head())

# Check for negative or zero prices
invalid_prices = df_clean[df_clean['Price'] <= 0]
print(f"\nRecords with invalid prices (≤ 0): {len(invalid_prices)}")
print(".1f")

if len(invalid_prices) > 0:
    print("\nSample invalid price records:")
    print(invalid_prices[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head())

# Check for extremely high quantities (potential data errors)
high_qty_threshold = df_clean['Quantity'].quantile(0.99)
high_qty_records = df_clean[df_clean['Quantity'] > high_qty_threshold]
print(f"\nRecords with quantity > 99th percentile ({high_qty_threshold}): {len(high_qty_records)}")

# Check for extremely high prices
high_price_threshold = df_clean['Price'].quantile(0.99)
high_price_records = df_clean[df_clean['Price'] > high_price_threshold]
print(f"Records with price > 99th percentile ({high_price_threshold:.2f}): {len(high_price_records)}")

# For analysis, we might want to keep returns but flag them
df_clean['IsReturn'] = df_clean['Quantity'] < 0

# Calculate total transaction value
df_clean['TotalValue'] = df_clean['Quantity'] * df_clean['Price']

print("\nTransaction value statistics:")
print(df_clean['TotalValue'].describe())

# Check for suspicious transactions (very high values)
high_value_threshold = df_clean['TotalValue'].quantile(0.99)
high_value_records = df_clean[df_clean['TotalValue'] > high_value_threshold]
print(f"\nRecords with transaction value > 99th percentile: {len(high_value_records)}")

=== OUTLIER DETECTION AND TREATMENT ===
Records with negative quantity: 18390
.1f

Sample negative quantity records:
     Invoice StockCode                    Description  Quantity  Price
178  C489449     22087       PAPER BUNTING WHITE LACE       -12   2.95
179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   1.65
180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   4.25
181  C489449     21896             POTTING SHED TWINE        -6   2.10
182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   2.95

Records with invalid prices (≤ 0): 70
.1f

Sample invalid price records:
      Invoice StockCode                     Description  Quantity  Price
4674   489825     22076              6 RIBBONS EMPIRE          12    0.0
6781   489998     48185             DOOR MAT FAIRY CAKE         2    0.0
16107  490727         M                          Manual         1    0.0
18738  490961     22065  CHRISTMAS PUDDING TRINKET POT          1    0.0
18739  490961     

In [10]:
# Data Validation and Business Rules
print("=== DATA VALIDATION AND BUSINESS RULES ===")

# Check invoice patterns - should start with digits or 'C' for cancellations
invoice_pattern_issues = df_clean[~df_clean['Invoice'].str.match(r'^[C]?[0-9]+$')]
print(f"Invoices not matching expected pattern: {len(invoice_pattern_issues)}")

if len(invoice_pattern_issues) > 0:
    print("Sample invoice pattern issues:")
    display(invoice_pattern_issues['Invoice'].head())

# Check StockCode patterns - should be alphanumeric
stockcode_issues = df_clean[~df_clean['StockCode'].str.match(r'^[A-Za-z0-9]+$')]
print(f"\nStockCodes with unusual characters: {len(stockcode_issues)}")

# Check for test/stock transactions (common in retail datasets)
test_transactions = df_clean[
    df_clean['StockCode'].str.contains('TEST', case=False) |
    df_clean['Description'].str.contains('TEST', case=False) |
    df_clean['StockCode'].str.contains('POSTAGE', case=False) |
    df_clean['Description'].str.contains('POSTAGE', case=False)
]
print(f"\nPotential test/postage transactions: {len(test_transactions)}")

# Check for manual transactions
manual_transactions = df_clean[df_clean['StockCode'].str.startswith('M')]
print(f"Manual transactions (StockCode starts with 'M'): {len(manual_transactions)}")

# Validate date consistency
# Check if there are future dates (beyond today's date)
future_dates = df_clean[df_clean['InvoiceDate'] > pd.Timestamp.now()]
print(f"\nRecords with future dates: {len(future_dates)}")

# Check for very old dates
old_dates = df_clean[df_clean['InvoiceDate'] < pd.Timestamp('2009-01-01')]
print(f"Records with dates before 2009: {len(old_dates)}")

# Country validation
print(f"\nUnique countries: {df_clean['Country'].nunique()}")
print("Top 10 countries by transaction count:")
display(df_clean['Country'].value_counts().head(10))

# Check for unusual countries or misspellings
unusual_countries = df_clean[~df_clean['Country'].str.match(r'^[A-Za-z\s\-\.]+$')]
if len(unusual_countries) > 0:
    print(f"\nCountries with unusual characters: {len(unusual_countries)}")
    display(unusual_countries['Country'].unique())

=== DATA VALIDATION AND BUSINESS RULES ===
Invoices not matching expected pattern: 0

StockCodes with unusual characters: 37

Potential test/postage transactions: 2015
Manual transactions (StockCode starts with 'M'): 1085

Records with future dates: 0
Records with dates before 2009: 0

Unique countries: 41
Top 10 countries by transaction count:


Country
United Kingdom    716115
Germany            17339
EIRE               16014
France             13897
Netherlands         5137
Spain               3754
Belgium             3110
Switzerland         3058
Portugal            2414
Australia           1890
Name: count, dtype: int64

In [11]:
# Feature Engineering for Cohort Analysis and Time Series
print("=== FEATURE ENGINEERING ===")

# Create date-related features for time series analysis
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek  # 0=Monday, 6=Sunday
df_clean['DayOfWeekName'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['MonthYear'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['Quarter'] = df_clean['InvoiceDate'].dt.quarter

# Create cohort analysis features
# First purchase date per customer
customer_first_purchase = df_clean.groupby('Customer ID')['InvoiceDate'].min().reset_index()
customer_first_purchase.columns = ['Customer ID', 'FirstPurchaseDate']

# Merge back to main dataframe
df_clean = df_clean.merge(customer_first_purchase, on='Customer ID', how='left')

# Cohort features
df_clean['CohortYear'] = df_clean['FirstPurchaseDate'].dt.year
df_clean['CohortMonth'] = df_clean['FirstPurchaseDate'].dt.month
df_clean['CohortQuarter'] = df_clean['FirstPurchaseDate'].dt.quarter
df_clean['CohortMonthYear'] = df_clean['FirstPurchaseDate'].dt.to_period('M')

# Calculate cohort index (months since first purchase)
df_clean['CohortIndex'] = ((df_clean['Year'] - df_clean['CohortYear']) * 12 +
                          (df_clean['Month'] - df_clean['CohortMonth']))

# Create positive value column (for LTV calculations, we might want to consider absolute values)
df_clean['PositiveQuantity'] = df_clean['Quantity'].abs()
df_clean['PositiveTotalValue'] = df_clean['PositiveQuantity'] * df_clean['Price']

# Invoice type classification
df_clean['InvoiceType'] = df_clean['Invoice'].apply(
    lambda x: 'Cancellation' if str(x).startswith('C') else 'Sale'
)

# Product category inference (basic)
df_clean['ProductCategory'] = df_clean['Description'].apply(
    lambda x: 'Gift' if 'gift' in str(x).lower() else
             'Decoration' if any(word in str(x).lower() for word in ['decoration', 'ornament', 'light']) else
             'Kitchen' if any(word in str(x).lower() for word in ['mug', 'cup', 'plate', 'kitchen']) else
             'Other'
)

print("✓ Feature engineering completed")
print(f"New features added: {len(df_clean.columns) - 8}")  # 8 original columns
print("\nNew columns:")
new_columns = [col for col in df_clean.columns if col not in
               ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
                'Price', 'Customer ID', 'Country']]
print(new_columns)

=== FEATURE ENGINEERING ===
✓ Feature engineering completed
New features added: 20

New columns:
['IsReturn', 'TotalValue', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'DayOfWeekName', 'MonthYear', 'Quarter', 'FirstPurchaseDate', 'CohortYear', 'CohortMonth', 'CohortQuarter', 'CohortMonthYear', 'CohortIndex', 'PositiveQuantity', 'PositiveTotalValue', 'InvoiceType', 'ProductCategory']


In [12]:
# Final Dataset Preparation and Summary
print("=== FINAL DATASET PREPARATION ===")

# Final data quality check
print("Final dataset summary:")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Memory usage: {df_clean.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

# Final missing values check
final_missing = df_clean.isnull().sum()
if final_missing.sum() > 0:
    print("\nRemaining missing values:")
    print(final_missing[final_missing > 0])
else:
    print("\n✓ No missing values remaining")

# Data types final check
print(f"\nData types:")
print(df_clean.dtypes)

# Key statistics for analysis
print(f"\nKey Statistics:")
print(f"- Unique customers: {df_clean['Customer ID'].nunique():,}")
print(f"- Unique products: {df_clean['StockCode'].nunique():,}")
print(f"- Date range: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
print(f"- Total transactions: {len(df_clean):,}")
print(f"- Total revenue: £{df_clean[df_clean['TotalValue'] > 0]['TotalValue'].sum():,.2f}")
print(f"- Returns/Cancellations: {df_clean['IsReturn'].sum():,}")

# Create separate datasets for different analyses
# 1. Full dataset (including returns)
df_full = df_clean.copy()

# 2. Sales only dataset (excluding returns/cancellations)
df_sales = df_clean[df_clean['Quantity'] > 0].copy()

# 3. Returns only dataset
df_returns = df_clean[df_clean['Quantity'] < 0].copy()

print(f"\nDataset splits:")
print(f"- Full dataset: {len(df_full):,} transactions")
print(f"- Sales only: {len(df_sales):,} transactions")
print(f"- Returns only: {len(df_returns):,} transactions")

# Save cleaned datasets
output_dir = '../data/cleaned/'
import os
os.makedirs(output_dir, exist_ok=True)

# Save as CSV for easier loading in other notebooks
df_full.to_csv(f'{output_dir}online_retail_cleaned_full.csv', index=False)
df_sales.to_csv(f'{output_dir}online_retail_cleaned_sales.csv', index=False)
df_returns.to_csv(f'{output_dir}online_retail_cleaned_returns.csv', index=False)

print(f"\n✓ Cleaned datasets saved to {output_dir}")
print("- online_retail_cleaned_full.csv: All transactions")
print("- online_retail_cleaned_sales.csv: Sales transactions only")
print("- online_retail_cleaned_returns.csv: Returns/cancellations only")

# Display sample of final dataset
print(f"\nSample of final cleaned dataset:")
display(df_full.head())

print("\n🎉 Data cleaning completed successfully!")
print("Ready for cohort analysis and time series forecasting.")

=== FINAL DATASET PREPARATION ===
Final dataset summary:
Shape: 797,885 rows × 28 columns
Memory usage: 217.77 MB

✓ No missing values remaining

Data types:
Invoice                          str
StockCode                        str
Description                   object
Quantity                       int64
InvoiceDate           datetime64[us]
Price                        float64
Customer ID                    int64
Country                          str
IsReturn                        bool
TotalValue                   float64
Year                           int32
Month                          int32
Day                            int32
Hour                           int32
DayOfWeek                      int32
DayOfWeekName                    str
MonthYear                  period[M]
Quarter                        int32
FirstPurchaseDate     datetime64[us]
CohortYear                     int32
CohortMonth                    int32
CohortQuarter                  int32
CohortMonthYear            p

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsReturn,TotalValue,Year,Month,Day,Hour,DayOfWeek,DayOfWeekName,MonthYear,Quarter,FirstPurchaseDate,CohortYear,CohortMonth,CohortQuarter,CohortMonthYear,CohortIndex,PositiveQuantity,PositiveTotalValue,InvoiceType,ProductCategory
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,2009,12,1,7,1,Tuesday,2009-12,4,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,83.4,Sale,Decoration
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,2009,12,1,7,1,Tuesday,2009-12,4,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,81.0,Sale,Decoration
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,2009,12,1,7,1,Tuesday,2009-12,4,2009-12-01 07:45:00,2009,12,4,2009-12,0,12,81.0,Sale,Decoration
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,2009,12,1,7,1,Tuesday,2009-12,4,2009-12-01 07:45:00,2009,12,4,2009-12,0,48,100.8,Sale,Other
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,2009,12,1,7,1,Tuesday,2009-12,4,2009-12-01 07:45:00,2009,12,4,2009-12,0,24,30.0,Sale,Other



🎉 Data cleaning completed successfully!
Ready for cohort analysis and time series forecasting.


In [13]:
# Manual save of cleaned datasets
print("=== MANUAL SAVE OF CLEANED DATA ===")

import os
output_dir = 'data/cleaned/'
os.makedirs(output_dir, exist_ok=True)

# Save as CSV for easier loading in other notebooks
df_full.to_csv(f'{output_dir}online_retail_cleaned_full.csv', index=False)
df_sales.to_csv(f'{output_dir}online_retail_cleaned_sales.csv', index=False)
df_returns.to_csv(f'{output_dir}online_retail_cleaned_returns.csv', index=False)

print(f"✓ Cleaned datasets saved to {output_dir}")
print("- online_retail_cleaned_full.csv: All transactions")
print("- online_retail_cleaned_sales.csv: Sales transactions only")
print("- online_retail_cleaned_returns.csv: Returns/cancellations only")

# Verify file sizes
for file in ['online_retail_cleaned_full.csv', 'online_retail_cleaned_sales.csv', 'online_retail_cleaned_returns.csv']:
    file_path = f'{output_dir}{file}'
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(".1f")
    else:
        print(f"✗ {file} was not saved")

=== MANUAL SAVE OF CLEANED DATA ===
✓ Cleaned datasets saved to data/cleaned/
- online_retail_cleaned_full.csv: All transactions
- online_retail_cleaned_sales.csv: Sales transactions only
- online_retail_cleaned_returns.csv: Returns/cancellations only
.1f
.1f
.1f


## Phase 0: Data Contract and Reproducibility Checks
Các kiểm tra dưới đây giúp khóa chuẩn dữ liệu đầu vào cho modeling:
- Đúng schema bắt buộc.
- Không null ở các cột trọng yếu.
- Kiểu thời gian hợp lệ.
- Cấu hình chạy được ghi nhận để trace kết quả.

In [ ]:
import pandas as pd

REQUIRED_COLS = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "TotalValue",
    "InvoiceType",
    "IsReturn",
]

missing_cols = [col for col in REQUIRED_COLS if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

critical_cols = ["InvoiceDate", "Customer ID", "TotalValue"]
null_report = df[critical_cols].isna().sum()
if (null_report > 0).any():
    raise ValueError(f"Critical null values detected:\n{null_report}")

if not pd.api.types.is_datetime64_any_dtype(df["InvoiceDate"]):
    raise TypeError("InvoiceDate must be datetime64 dtype.")

print("Data contract check: PASSED")
print(f"- Rows: {len(df):,}")
print(f"- Columns: {df.shape[1]}")
print(f"- Date range: {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")

In [ ]:
import random
import numpy as np
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if "df_sales" not in globals() or "df_returns" not in globals():
    if "df" not in globals():
        raise NameError("Expected dataframe 'df' not found. Run cleaning cells before this section.")
    df_sales = df[df["Quantity"] > 0].copy()
    df_returns = df[df["Quantity"] < 0].copy()

run_config = {
    "seed": SEED,
    "run_timestamp": datetime.utcnow().isoformat(timespec="seconds"),
    "dataset_rows": int(len(df)),
    "sales_rows": int(len(df_sales)),
    "return_rows": int(len(df_returns)),
    "unique_customers": int(df["Customer ID"].nunique()),
    "unique_invoices": int(df["Invoice"].nunique()),
}

print("Run config")
for k, v in run_config.items():
    print(f"- {k}: {v}")